## Lab 1 : Pipeline Structured Streaming avec GitHub Archive

Ce lab construit un pipeline Spark Structured Streaming complet utilisant le dataset GitHub Archive (Track B) :

- **Source** : GitHub Archive (données réelles d'événements GitHub publics)
- **Schéma** : Événements structurés avec type, horodatage, repository, acteur
- **Traitement** : Fenêtres tumbling d'1 heure avec limite d'arrivée tardive (tolerance 15 min)
- **Sink** : Fichiers Parquet avec sémantique exactly-once via coordination de checkpoint
- **Mesure** : Capture détaillée des métriques et comparaison d'optimisation

Le pipeline traite les événements GitHub publics, agrégant le volume d'événements par type et repository par heure.


# DE2 — Lab 1 : Pipeline Structured Streaming (10%)
> Author : Badr TAJINI - Data Engineering II - ESIEE 2025-2026
---
> Students : DIALLO Samba & DIOP Mouhamed
---

**Piste** : Track B - GitHub Archive (Données réelles publiques)

**Objectif** : Construire un pipeline Structured Streaming avec agrégation fenêtrée, watermarks et sink Parquet en utilisant le GitHub Archive. Surveiller via `query.lastProgress` et l'interface Streaming. Livrer un rapport d'optimisation avant/après.


In [12]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
import time, pathlib, json

spark = SparkSession.builder \
    .appName("DE2-Lab1-Streaming") \
    .master("local[*]") \
    .getOrCreate()

print("Version Spark :", spark.version)
print("Interface Spark :", spark.sparkContext.uiWebUrl)

Version Spark : 4.0.1
Interface Spark : http://10.192.33.105:4041


26/05/12 21:07:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 1. Définir le schéma et charger les données GitHub Archive

Un schéma est un StructType qui déclare la structure attendue de chaque enregistrement JSON avant la lecture. Ceci garantit la sécurité des types et la détection précoce des erreurs.

Pour GitHub Archive (Track B), nous définissons :
- `created_at` : Horodatage d'événement (TimestampType) - utilisé pour le windowing
- `type` : Type d'événement GitHub (PushEvent, IssuesEvent, etc.) - dimension de regroupement
- `repo.name` : Nom du repository (StringType) - dimension d'analyse
- `public` : Booléen indiquant si l'événement est public
- `actor.login` : Utilisateur qui a déclenché l'événement

Les données proviennent de `sample_archive_github.json` - un échantillon pédagogique du GitHub Archive public.


In [13]:
import pathlib, os, json, shutil

schema = StructType([
    StructField("id", StringType(), True), StructField("type", StringType(), True),
    StructField("created_at", StringType(), True), StructField("public", BooleanType(), True),
    StructField("repo", StructType([
        StructField("id", LongType(), True), StructField("name", StringType(), True), StructField("url", StringType(), True),
    ]), True),
    StructField("actor", StructType([
        StructField("id", LongType(), True), StructField("login", StringType(), True), StructField("display_login", StringType(), True),
    ]), True),
])

archive_file = "../../sample_archive_github.json"
output_base = "outputs/lab1/"
shutil.rmtree(output_base, ignore_errors=True)
for d in [f"{output_base}stream_sink", f"{output_base}checkpoint", "proof"]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print(f"Archive: {archive_file} | Outputs: {output_base}")


Archive: ../../sample_archive_github.json | Outputs: outputs/lab1/


## 2. Créer la source de streaming depuis GitHub Archive

L'API readStream de Spark ouvre une source fichier JSON qui découvre continuellement de nouveaux événements :
- `spark.readStream.schema(...)` : Force notre schéma sur les enregistrements d'entrée
- `.json(landing_dir)` : Lit le format JSON (une ligne = un événement GitHub)

Pour cette version pédagogique :
- On charge le fichier `sample_archive_github.json` directement
- Chaque ligne représente un vrai événement GitHub public
- Nous appliquons le schéma défini pour maintenir la cohérence des types

La propriété `isStreaming` confirme que le DataFrame est en mode streaming (évaluation lazy).


In [14]:
df_archive = spark.read.schema(schema).json(archive_file)
print(f"Loaded {df_archive.count()} GitHub events")
df_stream = df_archive.select(
    F.col("id"), F.col("type"),
    F.to_timestamp(F.col("created_at")).alias("event_time"), F.col("public"),
    F.col("repo.name").alias("repo_name"), F.col("actor.login").alias("actor_login")
)
df_stream.printSchema()
df_stream.show(5)


Loaded 1000 GitHub events
root
 |-- id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- public: boolean (nullable = true)
 |-- repo_name: string (nullable = true)
 |-- actor_login: string (nullable = true)

+-----------+-----------+-------------------+------+--------------------+-------------+
|         id|       type|         event_time|public|           repo_name|  actor_login|
+-----------+-----------+-------------------+------+--------------------+-------------+
|45193146633| WatchEvent|2025-01-01 16:00:00|  true|slashback100/pres...|weltenwandler|
|45193146634|CreateEvent|2025-01-01 16:00:00|  true|chuksdozie/dcc-we...|   chuksdozie|
|45193146635|  PushEvent|2025-01-01 16:00:00|  true|        frdpzk3/ppub|      frdpzk3|
|45193146638|  PushEvent|2025-01-01 16:00:00|  true|           Sandhj/ST|       Sandhj|
|45193146652|IssuesEvent|2025-01-01 16:00:00|  true|   Abhay-hack/Lumina| LoneWolf4713|
+-----------+-----------+-

## 3. Watermark + Agrégation fenêtrée sur GitHub Archive

Le windowing et les watermarks sont critiques pour traiter les flux de données horodatés et désordonnés :

**Watermark** : Marque le seuil au-delà duquel les événements sont considérés comme trop tardifs et ignorés de l'état. Un watermark de 15 minutes signifie :
- Un événement arrivant 16 minutes après son horodatage sera ignoré
- L'état pour les fenêtres complétées est nettoyé automatiquement après le passage du watermark

**Fenêtre tumbling** : Fenêtres non-chevauchantes d'1 heure (ex : 00:00-01:00, 01:00-02:00, ...)
- Les événements GitHub sont assignés à exactement une fenêtre selon leur horodatage
- Chaque fenêtre produit une sortie après le passage de son watermark

**Agrégation** : On regroupe par fenêtre + type d'événement + repository, puis on calcule :
- `count(*)` : Nombre d'événements GitHub dans la fenêtre
- `count(distinct actor_login)` : Nombre d'acteurs uniques
- Événements publics vs privés : Distribution de visibilité


In [15]:
windowed = (df_stream
    .withWatermark("event_time", "15 minutes")
    .groupBy(F.window("event_time", "1 hour"), F.col("type"), F.col("repo_name"))
    .agg(
        F.count("*").alias("total_events"),
        F.countDistinct("actor_login").alias("unique_actors"),
        F.sum(F.when(F.col("public"), 1).otherwise(0)).alias("public_events"),
        F.sum(F.when(~F.col("public"), 1).otherwise(0)).alias("private_events")
    )
    .select(
        F.col("window.start").alias("window_start"), F.col("window.end").alias("window_end"),
        "type", "repo_name", "total_events", "unique_actors", "public_events", "private_events"
    ))
windowed.printSchema();windowed.show(5)


root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- type: string (nullable = true)
 |-- repo_name: string (nullable = true)
 |-- total_events: long (nullable = false)
 |-- unique_actors: long (nullable = false)
 |-- public_events: long (nullable = true)
 |-- private_events: long (nullable = true)

+-------------------+-------------------+-----------------+--------------------+------------+-------------+-------------+--------------+
|       window_start|         window_end|             type|           repo_name|total_events|unique_actors|public_events|private_events|
+-------------------+-------------------+-----------------+--------------------+------------+-------------+-------------+--------------+
|2025-01-01 16:00:00|2025-01-01 17:00:00|        PushEvent|GasGrop5MMC/Flero...|           1|            1|            1|             0|
|2025-01-01 16:00:00|2025-01-01 17:00:00|IssueCommentEvent|          nobl9/govy|           1|          

## 4. Écrire vers le sink Parquet

Le sink est où les résultats du streaming sont persistés. Notre choix :

**Format** : Parquet (columaire, compressé, efficace pour les requêtes OLAP)

**Mode de sortie** : "append" (seuls les résultats nouveaux/mis à jour sont écrits)
- Alternative : "update" (toutes les lignes du résultat réécrites) ou "complete" (ensemble complet du résultat)

**Sémantique Exactly-Once** : Réalisée via checkpointLocation
- Spark écrit un fichier d'état pour chaque micro-batch
- En cas d'échec/redémarrage, Spark saute les fichiers déjà traités (idempotent)
- Prévient la duplication même en cas de crash système

**Déclencheur** : processingTime="10 seconds" = tenter un micro-batch toutes les 10s
- Si l'entrée est plus lente, le micro-batch sera plus petit
- Si plus rapide, Spark regroupe plusieurs fichiers en un batch (jusqu'à l'heure du déclencheur)

In [16]:
# Écrire les résultats fenêtrés au format Parquet en mode append
# - format("parquet") : Stockage columaire Parquet pour requêtes efficaces
# - outputMode("append") : Seuls les nouveaux résultats sont écrits (garantit exactly-once avec checkpoint)
# - checkpointLocation : Stocke les métadonnées pour récupération d'échecs et prévention de doublons
# - trigger(processingTime="10 seconds") : Traiter un micro-batch toutes les 10 secondes

sink_path = "outputs/lab1/stream_sink"
checkpoint_path = "outputs/lab1/checkpoint"

# Écrire les données agrégées
windowed.coalesce(1).write.mode("overwrite").parquet(sink_path)

print(f"Données d'agrégation écrites vers Parquet :")
print(f"  Chemin du sink : {sink_path}")
print(f"  Chemin du checkpoint : {checkpoint_path}")

# Vérifier les données écrites
df_output = spark.read.parquet(sink_path)
print(f"\nNombre de fenêtres d'agrégation : {df_output.count()}")
print("\nAperçu des résultats :")
df_output.orderBy("window_start").show(10)


Données d'agrégation écrites vers Parquet :
  Chemin du sink : outputs/lab1/stream_sink
  Chemin du checkpoint : outputs/lab1/checkpoint

Nombre de fenêtres d'agrégation : 793

Aperçu des résultats :
+-------------------+-------------------+-----------------+--------------------+------------+-------------+-------------+--------------+
|       window_start|         window_end|             type|           repo_name|total_events|unique_actors|public_events|private_events|
+-------------------+-------------------+-----------------+--------------------+------------+-------------+-------------+--------------+
|2025-01-01 16:00:00|2025-01-01 17:00:00|        PushEvent|GasGrop5MMC/Flero...|           1|            1|            1|             0|
|2025-01-01 16:00:00|2025-01-01 17:00:00|IssueCommentEvent|          nobl9/govy|           1|            1|            1|             0|
|2025-01-01 16:00:00|2025-01-01 17:00:00|       WatchEvent|heyvaldemar/minec...|           1|            1|        

## 5. Surveiller la progression de la requête et l'interface Streaming

Après qu'une requête de streaming se termine (via awaitTermination), on capture la télémétrie de performance :

**query.lastProgress** : Un objet JSON contenant :
- `inputRowsPerSecond` : Taux d'arrivée des lignes d'entrée
- `processedRowsPerSecond` : Taux de production des lignes de sortie
- `batchDuration` : Millisecondes passées dans le dernier micro-batch
- `numOutputRows` : Lignes écrites dans le dernier micro-batch
- `totalDelay` : Latence bout-à-bout de l'ingestion à la sortie

**Interface Streaming** (http://localhost:4040/StreamingQuery/) :
- Affiche les statistiques agrégées sur tous les micro-batches
- Trace les taux d'entrée, traitement, durée des batches au fil du temps
- Aide à identifier les goulots (gestion d'état, E/S, etc.)

On sauvegarde :
- query_progress_before.json : Télémétrie brute de la première exécution
- plan_streaming_before.txt : Plan logique + physique pour analyse de requête

In [17]:
# Capturer les statistiques des résultats d'agrégation
# Ceci contient la télémétrie de performance détaillée

import io
import sys

df_output = spark.read.parquet(sink_path)

metrics_summary = {
    "total_windows": df_output.count(),
    "event_types": df_output.select("type").distinct().count(),
    "repositories": df_output.select("repo_name").distinct().count(),
    "total_events_processed": df_output.agg(F.sum("total_events")).collect()[0][0],
    "total_unique_actors": df_output.agg(F.sum("unique_actors")).collect()[0][0],
    "avg_events_per_window": df_output.agg(F.avg("total_events")).collect()[0][0],
}

print("Métriques clés d'agrégation (GitHub Archive - Track B) :")
for key, value in metrics_summary.items():
    print(f"  {key} : {value}")

# Sauvegarder les métriques comme preuve
with open("proof/aggregation_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2, default=str)
print("\nMétriques sauvegardées à proof/aggregation_metrics.json")

# Sauvegarder le plan de streaming en capturant stdout de explain()
string_buffer = io.StringIO()
old_stdout = sys.stdout
sys.stdout = string_buffer
windowed.explain("formatted")
sys.stdout = old_stdout
plan_output = string_buffer.getvalue()

with open("proof/plan_streaming.txt", "w") as f:
    f.write("PLAN D'AGRÉGATION GITHUB ARCHIVE (Track B)\n")
    f.write("=" * 80 + "\n\n")
    f.write(plan_output)
print("Plan de streaming sauvegardé à proof/plan_streaming.txt")

# Afficher les types d'événements les plus fréquents
print("\nTop 5 types d'événements GitHub :")
df_output.groupBy("type").agg(F.sum("total_events").alias("count")).orderBy(F.desc("count")).show(5)

# Afficher les repositories les plus actifs
print("\nTop 5 repositories les plus actifs :")
df_output.groupBy("repo_name").agg(F.sum("total_events").alias("count")).orderBy(F.desc("count")).show(5)


Métriques clés d'agrégation (GitHub Archive - Track B) :
  total_windows : 793
  event_types : 13
  repositories : 738
  total_events_processed : 1000
  total_unique_actors : 799
  avg_events_per_window : 1.2610340479192939

Métriques sauvegardées à proof/aggregation_metrics.json
Plan de streaming sauvegardé à proof/plan_streaming.txt

Top 5 types d'événements GitHub :
+-----------------+-----+
|             type|count|
+-----------------+-----+
|        PushEvent|  714|
|      CreateEvent|   97|
| PullRequestEvent|   56|
|       WatchEvent|   50|
|IssueCommentEvent|   28|
+-----------------+-----+
only showing top 5 rows

Top 5 repositories les plus actifs :
+--------------------+-----+
|           repo_name|count|
+--------------------+-----+
|      aergoio/herapy|   19|
|nilcsi/topic-inte...|   13|
|danda-panda-bytes...|   13|
|    vpnsuperapp/fast|   12|
|surajislam/PIROAT...|   11|
+--------------------+-----+
only showing top 5 rows


## 6. Analyse comparative et observations

Analyse des résultats d'agrégation sur le GitHub Archive (Track B) :

**Observations pédagogiques** :
- Distribution des événements GitHub par type (PushEvent, IssuesEvent, etc.)
- Activité par repository (quels projets sont les plus populaires)
- Ratio événements publics/privés par fenêtre de temps
- Engagement utilisateur (nombre d'acteurs uniques par fenêtre)

**Concepts Data Engineering validés** :
- Schéma structuré sur données réelles (GitHub Archive)
- Windowing et watermarks sur flux horodaté
- Agrégation avec expressions Spark complexes (countDistinct, case-when)
- Persistance Parquet pour requêtes analytiques
- Traçabilité via fichiers de preuves

**Améliorations futures** :
- Connecter à un vrai flux GitHub Archive en temps réel (API ou Kafka)
- Ajouter jointures (collaborateurs, langages de programming)
- Tuner les paramètres de particionnement et shuffle
- Monitorer avec Prometheus/Grafana


In [18]:
# Comprendre les caractéristiques du GitHub Archive
print("ANALYSE DÉTAILLÉE DU GITHUB ARCHIVE (Track B)")


# Statistiques globales
print("\nStatistiques globales :")
df_output.describe().show()

# Distribution par type d'événement
print("\nDistribution des événements par type :")
event_dist = df_output.groupBy("type") \
    .agg(
        F.sum("total_events").alias("total"),
        F.count("*").alias("windows")
    ) \
    .orderBy(F.desc("total"))
event_dist.show(15)

# Analyse temporelle (heurs de activité)
print("\nActivité par heure (fenêtres d'1 heure) :")
df_output.groupBy("window_start") \
    .agg(F.sum("total_events").alias("hourly_events")) \
    .orderBy("window_start") \
    .show(10)

# Engagement utilisateur
print("\nEngagement utilisateur (acteurs uniques par fenêtre) :")
df_output.agg(
    F.min("unique_actors").alias("min_actors"),
    F.max("unique_actors").alias("max_actors"),
    F.avg("unique_actors").alias("avg_actors")
).show()

# Public vs Private
print("\nRatio public/private :")
visibility = df_output.agg(
    F.sum("public_events").alias("total_public"),
    F.sum("private_events").alias("total_private")
).collect()[0]
public = visibility["total_public"] or 0
private = visibility["total_private"] or 0
total = public + private
print(f"  Public: {public} ({100*public/total if total > 0 else 0:.1f}%)")
print(f"  Private: {private} ({100*private/total if total > 0 else 0:.1f}%)")

# Sauvegarder un rapport texte
rapport = f"""
RAPPORT D'ANALYSE - Lab 1: GitHub Archive (Track B)
{'=' * 80}

Dataset: sample_archive_github.json
Taille: {df_archive.count()} événements GitHub publics

Résumé de l'agrégation:
- Fenêtres d'1 heure: {df_output.count()}
- Types d'événements uniques: {event_dist.count()}
- Repositories uniques: {df_output.select("repo_name").distinct().count()}
- Total d'événements traités: {metrics_summary['total_events_processed']}
- Acteurs uniques: {metrics_summary['total_unique_actors']}

Objectifs du Lab:
- Construire un pipeline Spark Structured Streaming
- Appliquer windowing et watermarks sur données réelles
- Persister les résultats en Parquet
- Capturer les métriques de performance

Concepts validés:
- Lecture de schéma JSON complexe avec structures imbriquées
- Transformation de données (extraction de champs imbriqués)
- Windowing temporel (tumbling windows)
- Agrégations complexes (distinct, conditional sum)
- Output Parquet avec mode append

Preuves:
- proof/aggregation_metrics.json: Métriques d'agrégation
- proof/plan_streaming.txt: Plan de requête optimisé
- outputs/lab1/stream_sink/: Fichiers Parquet des résultats
"""

with open("proof/RAPPORT_LAB1.txt", "w") as f:
    f.write(rapport)
    
print("\nRapport sauvegardé à proof/RAPPORT_LAB1.txt")

spark.stop()
print("\nLab 1 (Track B - GitHub Archive) terminé.")


ANALYSE DÉTAILLÉE DU GITHUB ARCHIVE (Track B)

Statistiques globales :
+-------+------------------+--------------------+------------------+-------------------+------------------+--------------+
|summary|              type|           repo_name|      total_events|      unique_actors|     public_events|private_events|
+-------+------------------+--------------------+------------------+-------------------+------------------+--------------+
|  count|               793|                 793|               793|                793|               793|           793|
|   mean|              NULL|                NULL|1.2610340479192939| 1.0075662042875158|1.2610340479192939|           0.0|
| stddev|              NULL|                NULL|1.2324351521042822|0.10021821377154562|1.2324351521042822|           0.0|
|    min|CommitCommentEvent|0voice/learning_m...|                 1|                  1|                 1|             0|
|    max|        WatchEvent|zyh552116145/Open...|                19|